In [1]:
##### Calculates capital and labor intensities 
# for countries and sub-national regions 

import os
import numpy as np
import pandas as pd
import geopandas as gpd

In [2]:
##### Load data

# Get the current working directory
cd = os.path.dirname(os.getcwd())

### Load sub-national data 

# load capital data
capital = pd.read_csv(f"{cd}/Data/Clean/Capital_stock/subnational_capital_stock_final.csv")

# load labor data
labor = pd.read_csv(f"{cd}/Data/Clean/Labor/subnational_labor_final.csv")

# load production data
production_capital = pd.read_csv(f"{cd}/Data/Clean/Production/Subnational_stats/subnational_production_stats_capital_regions_target.csv")
production_labor = pd.read_csv(f"{cd}/Data/Clean/Production/Subnational_stats/subnational_production_stats_labor_regions_target.csv")

# save path
capital_path = f"{cd}/Data/Clean/Intensities/subnational_capital_intensity.csv"
labor_path = f"{cd}/Data/Clean/Intensities/subnational_labor_intensity.csv"

### Load country data

# load capital data
country_capital = pd.read_csv(f"{cd}/Data/Clean/Capital_stock/FAO_capital_stock_adjusted.csv")

# load labor data
country_labor = pd.read_csv(f"{cd}/Data/Clean/Labor/ILO_ag_labor_estimate_adjusted.csv")

# load production data
country_production = pd.read_csv(f"{cd}/Data/Clean/Production/FAO_data/FAO_production_value_clean_total.csv")

# save path
country_path = f"{cd}/Data/Clean/Intensities/country_intensities.csv"


In [3]:
##### Calculate country intensities 

# merge 
country_intensities = country_capital.merge(country_labor, on='ISO3', how='outer')
country_intensities = country_intensities.merge(country_production, on='ISO3', how='inner')

# calculate intensities 
country_intensities['labor_intensity_jobs_per_million_USD'] = (country_intensities['ag_labor_thousands_2020'] * 1000) / (country_intensities['ag_production_thousand_USD_2020'] / 1000)
country_intensities['capital_intensity_USD_per_USD'] = (country_intensities['ag_capital_stock_mil_USD_nominal'] * 1e6) / (country_intensities['ag_production_thousand_USD_2020'] * 1e3)

# save
country_intensities.to_csv(country_path, index=False)

In [4]:
##### Calculate sub-national intensities 

# merge
sub_national_capital = capital.merge(production_capital, on='PROJ_ID', how='outer')
sub_national_labor = labor.merge(production_labor, on='PROJ_ID', how='outer')

# drop missing 
sub_national_capital = sub_national_capital.dropna()
sub_national_labor = sub_national_labor.dropna()

# filter out where produciton or capital/labor is 0
sub_national_capital = sub_national_capital[~(sub_national_capital['ag_capital_stock_USD_nominal'] == 0)]
sub_national_capital = sub_national_capital[~(sub_national_capital['total_production_USD'] == 0)]

sub_national_labor = sub_national_labor[~(sub_national_labor['ag_jobs'] == 0)]
sub_national_labor = sub_national_labor[~(sub_national_labor['total_production_USD'] == 0)]

# calculate intensities 
sub_national_capital['capital_intensity_USD_per_USD'] = sub_national_capital['ag_capital_stock_USD_nominal'] / sub_national_capital['total_production_USD']
sub_national_labor['labor_intensity_jobs_per_million_USD'] = sub_national_labor['ag_jobs'] / (sub_national_labor['total_production_USD'] / 1e6)

# filter columns
cap_keep = ['PROJ_ID', 'GEO_ID_NAME', 'ag_capital_stock_USD_nominal', 'total_production_USD', 'capital_intensity_USD_per_USD']
lab_keep = ['PROJ_ID', 'GEO_ID_NAME', 'ag_jobs', 'total_production_USD', 'labor_intensity_jobs_per_million_USD']

sub_national_capital = sub_national_capital[cap_keep]
sub_national_labor = sub_national_labor[lab_keep]

# save
sub_national_capital.to_csv(capital_path, index=False)
sub_national_labor.to_csv(labor_path, index=False)